In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="rl-advanced-algorithms",
    notebook_name="01_trust_region_methods_experiments.ipynb"
)

# Experiments: Trust Region Methods

This notebook produces **runnable evidence** for the claims made in the trust region methods concept notebook and interview deep-dive. Every cell produces output that could be shown to an interviewer.

**What we test:**
1. KL divergence grows with learning rate — larger steps cause larger policy shifts
2. Policy collapse without trust regions — large learning rates can destroy performance irreversibly
3. Clipping reduces KL divergence — PPO-clip bounds policy change without explicit KL constraints

All experiments use a simple 10-state chain MDP defined inline. No gym dependency.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)
print("Setup complete.")

---
## Environment: 10-State Chain MDP

A simple MDP that is easy to reason about:

```
  [0] — [1] — [2] — [3] — [4] — [5] — [6] — [7] — [8] — [9]
                                                          +10 reward
```

- **States:** 0 through 9
- **Actions:** 0 = left, 1 = right
- **Reward:** +10 at state 9 (terminal), -0.1 per step elsewhere
- **Episode ends:** when the agent reaches state 9 or after 50 steps

The optimal policy is to always go right.

In [ ]:
class ChainMDP:
    """10-state chain MDP. Go right to reach the goal at state 9."""

    def __init__(self, n_states=10, goal_reward=10.0, step_penalty=-0.1, max_steps=50):
        self.n_states = n_states
        self.n_actions = 2  # 0 = left, 1 = right
        self.goal_reward = goal_reward
        self.step_penalty = step_penalty
        self.max_steps = max_steps
        self.state = 0
        self.steps = 0

    def reset(self):
        self.state = 0
        self.steps = 0
        return self.state

    def step(self, action):
        self.steps += 1
        if action == 1:  # right
            self.state = min(self.state + 1, self.n_states - 1)
        else:  # left
            self.state = max(self.state - 1, 0)

        if self.state == self.n_states - 1:
            return self.state, self.goal_reward, True
        elif self.steps >= self.max_steps:
            return self.state, self.step_penalty, True
        else:
            return self.state, self.step_penalty, False


# Quick sanity check: an agent that always goes right should reach state 9 in 9 steps
env = ChainMDP()
s = env.reset()
total_reward = 0.0
trajectory = [s]
for _ in range(20):
    s, r, done = env.step(1)  # always go right
    total_reward += r
    trajectory.append(s)
    if done:
        break

print(f"Trajectory (always right): {trajectory}")
print(f"Total reward: {total_reward:.1f}")
print(f"Steps taken: {env.steps}")
print(f"Reached goal: {s == 9}")
print()

# An agent that always goes left should never reach the goal
env.reset()
total_reward_left = 0.0
for _ in range(50):
    s, r, done = env.step(0)  # always go left
    total_reward_left += r
    if done:
        break

print(f"Total reward (always left): {total_reward_left:.1f}")
print(f"Reached goal: {s == 9}")

### Helper: softmax policy and trajectory collection

We use a simple linear softmax policy: for each state, we have logits for each action. The policy samples actions from a softmax over these logits.

In [ ]:
def get_policy_probs(logits):
    """Convert logits (n_states, n_actions) to action probabilities via softmax."""
    # logits: (n_states, n_actions)
    probs = torch.softmax(logits, dim=-1)
    return probs


def collect_trajectory(env, logits, max_steps=50):
    """
    Collect one trajectory using the softmax policy defined by logits.

    Returns:
        states: list of visited states
        actions: list of actions taken
        rewards: list of rewards received
    """
    probs = get_policy_probs(logits).detach().numpy()
    states, actions, rewards = [], [], []
    s = env.reset()
    for _ in range(max_steps):
        a = np.random.choice(env.n_actions, p=probs[s])
        states.append(s)
        actions.append(a)
        s_next, r, done = env.step(a)
        rewards.append(r)
        s = s_next
        if done:
            break
    return states, actions, rewards


def compute_returns(rewards, gamma=0.99):
    """Compute discounted returns from a list of rewards."""
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return returns


def compute_kl_divergence(old_logits, new_logits):
    """
    Compute mean KL(pi_old || pi_new) across all states.

    KL(P || Q) = sum_a P(a) * log(P(a) / Q(a))
    """
    old_probs = torch.softmax(old_logits, dim=-1)
    new_probs = torch.softmax(new_logits, dim=-1)
    # KL divergence per state, then average
    kl_per_state = (old_probs * (torch.log(old_probs + 1e-10) - torch.log(new_probs + 1e-10))).sum(dim=-1)
    return kl_per_state.mean().item()


# Sanity check: KL between identical logits should be 0
test_logits = torch.randn(10, 2)
kl_same = compute_kl_divergence(test_logits, test_logits)
print(f"KL(same, same) = {kl_same:.6f}  (should be ~0)")

# KL between different logits should be > 0
test_logits2 = torch.randn(10, 2)
kl_diff = compute_kl_divergence(test_logits, test_logits2)
print(f"KL(different, different) = {kl_diff:.6f}  (should be > 0)")

---
## Experiment 1: KL Divergence vs Learning Rate

**Claim being tested:** Larger learning rates produce larger KL divergence between old and new policies after a single gradient update.

**Why this matters in an interview:** This is the core justification for trust regions. If you can show that a single vanilla gradient step with a large learning rate moves the policy far away (in KL terms), you have explained *why* TRPO and PPO exist.

**Setup:**
- Initialize a random softmax policy over the 10-state chain
- Collect a batch of trajectories
- For each learning rate in [0.001, 0.01, 0.1, 0.5], compute one REINFORCE gradient update
- Measure KL(old || new) after the update
- Plot learning rate vs KL divergence

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

learning_rates = [0.001, 0.01, 0.1, 0.5]
n_trajectories = 20  # collect this many trajectories per LR measurement
env = ChainMDP()

# Use the same initial logits for all LR experiments so the comparison is fair
init_logits = torch.randn(10, 2) * 0.1  # small random init, near-uniform policy

kl_values = []

for lr in learning_rates:
    # Fresh copy of the initial logits
    logits = init_logits.clone().requires_grad_(True)
    old_logits = init_logits.clone().detach()

    # Collect trajectories under the current (old) policy
    all_states, all_actions, all_returns = [], [], []
    for _ in range(n_trajectories):
        states, actions, rewards = collect_trajectory(env, logits)
        rets = compute_returns(rewards)
        all_states.extend(states)
        all_actions.extend(actions)
        all_returns.extend(rets)

    # Compute REINFORCE loss
    states_t = torch.tensor(all_states, dtype=torch.long)
    actions_t = torch.tensor(all_actions, dtype=torch.long)
    returns_t = torch.tensor(all_returns, dtype=torch.float32)
    # Normalize returns for stability
    returns_t = (returns_t - returns_t.mean()) / (returns_t.std() + 1e-8)

    probs = torch.softmax(logits, dim=-1)
    action_probs = probs[states_t, actions_t]
    log_probs = torch.log(action_probs + 1e-10)
    loss = -(log_probs * returns_t).mean()

    # One gradient step
    loss.backward()
    with torch.no_grad():
        logits_new = logits - lr * logits.grad  # gradient descent on loss = ascent on objective

    # Measure KL divergence
    kl = compute_kl_divergence(old_logits, logits_new)
    kl_values.append(kl)

    print(f"LR = {lr:>5.3f}  |  KL(old || new) = {kl:.6f}")

print()
print(f"Ratio of largest to smallest KL: {kl_values[-1] / (kl_values[0] + 1e-12):.1f}x")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(learning_rates, kl_values, 'o-', linewidth=2, markersize=8, color='#1565c0')

# Annotate each point
for lr, kl in zip(learning_rates, kl_values):
    ax.annotate(f"{kl:.4f}", (lr, kl), textcoords="offset points",
                xytext=(10, 10), fontsize=9)

# Typical TRPO constraint line
ax.axhline(y=0.01, color='green', linestyle='--', linewidth=1.5, label=r'TRPO constraint $\delta = 0.01$')

ax.set_xlabel('Learning Rate', fontsize=12)
ax.set_ylabel('KL Divergence (old || new)', fontsize=12)
ax.set_title('Experiment 1: KL Divergence Grows with Learning Rate', fontsize=13, fontweight='bold')
ax.set_xscale('log')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observation: KL divergence increases with learning rate.")
print("Large LR (0.5) moves the policy far from the old one in a single step.")
print("This is exactly the problem trust regions are designed to prevent.")

### What the output shows

KL divergence grows with learning rate. A learning rate of 0.5 produces a KL divergence orders of magnitude larger than 0.001.

**The one sentence to say in an interview:** "Larger gradient steps move the policy further in KL space, which is why we need trust regions to bound the step size and prevent the data distribution from shifting too far."

---
## Experiment 2: Policy Collapse Without Trust Regions

**Claim being tested:** A very large learning rate can cause policy collapse — performance crashes and does not recover — because a bad policy generates bad data, which makes the next policy even worse.

**Why this matters in an interview:** This is the failure mode that trust region methods are built to prevent. Showing it happening with real numbers makes the argument concrete.

**Setup:**
- Train a REINFORCE agent on the chain MDP with LR = 0.5 (large) and LR = 0.01 (moderate)
- Run 100 training episodes for each
- Track total reward per episode
- The large-LR agent should show unstable or collapsing performance

In [ ]:
def train_reinforce(env, lr, n_episodes=100, seed=42):
    """
    Train a REINFORCE agent on the ChainMDP.

    Returns:
        episode_rewards: list of total rewards per episode
        kl_per_episode: list of KL divergence between consecutive policies
    """
    np.random.seed(seed)
    torch.manual_seed(seed)

    logits = torch.zeros(env.n_states, env.n_actions, requires_grad=True)
    episode_rewards = []
    kl_per_episode = []

    for ep in range(n_episodes):
        old_logits = logits.clone().detach()

        # Collect one trajectory
        states, actions, rewards = collect_trajectory(env, logits)
        total_reward = sum(rewards)
        episode_rewards.append(total_reward)

        # Compute returns
        returns = compute_returns(rewards)
        states_t = torch.tensor(states, dtype=torch.long)
        actions_t = torch.tensor(actions, dtype=torch.long)
        returns_t = torch.tensor(returns, dtype=torch.float32)
        returns_t = (returns_t - returns_t.mean()) / (returns_t.std() + 1e-8)

        # REINFORCE gradient
        probs = torch.softmax(logits, dim=-1)
        action_probs = probs[states_t, actions_t]
        log_probs = torch.log(action_probs + 1e-10)
        loss = -(log_probs * returns_t).mean()

        if logits.grad is not None:
            logits.grad.zero_()
        loss.backward()

        with torch.no_grad():
            logits_new = logits - lr * logits.grad
            kl = compute_kl_divergence(old_logits, logits_new)
            kl_per_episode.append(kl)
            # Apply the update
            logits.copy_(logits_new)
            logits.requires_grad_(True)

    return episode_rewards, kl_per_episode


env = ChainMDP()

print("Training with LR = 0.5 (large) ...")
rewards_large, kl_large = train_reinforce(env, lr=0.5, n_episodes=100, seed=42)
print(f"  Final 10-episode avg reward: {np.mean(rewards_large[-10:]):.2f}")
print(f"  Mean KL per update: {np.mean(kl_large):.4f}")

print("\nTraining with LR = 0.01 (moderate) ...")
rewards_moderate, kl_moderate = train_reinforce(env, lr=0.01, n_episodes=100, seed=42)
print(f"  Final 10-episode avg reward: {np.mean(rewards_moderate[-10:]):.2f}")
print(f"  Mean KL per update: {np.mean(kl_moderate):.4f}")

print(f"\nLarge LR has {np.mean(kl_large)/np.mean(kl_moderate):.1f}x higher mean KL divergence per update.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Learning curves
ax1 = axes[0]
window = 5

def smooth(values, w):
    """Simple moving average for smoothing."""
    if len(values) < w:
        return values
    cumsum = np.cumsum(np.insert(values, 0, 0))
    return (cumsum[w:] - cumsum[:-w]) / w

smoothed_large = smooth(rewards_large, window)
smoothed_moderate = smooth(rewards_moderate, window)

ax1.plot(rewards_large, alpha=0.2, color='red')
ax1.plot(range(window - 1, len(rewards_large)), smoothed_large,
         linewidth=2, color='red', label=f'LR = 0.5 (large)')

ax1.plot(rewards_moderate, alpha=0.2, color='blue')
ax1.plot(range(window - 1, len(rewards_moderate)), smoothed_moderate,
         linewidth=2, color='blue', label=f'LR = 0.01 (moderate)')

ax1.set_xlabel('Episode', fontsize=11)
ax1.set_ylabel('Total Reward', fontsize=11)
ax1.set_title('Experiment 2: Policy Collapse with Large LR', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Right: KL divergence per episode
ax2 = axes[1]
ax2.plot(kl_large, alpha=0.4, color='red')
smoothed_kl_large = smooth(kl_large, window)
ax2.plot(range(window - 1, len(kl_large)), smoothed_kl_large,
         linewidth=2, color='red', label='LR = 0.5')

ax2.plot(kl_moderate, alpha=0.4, color='blue')
smoothed_kl_moderate = smooth(kl_moderate, window)
ax2.plot(range(window - 1, len(kl_moderate)), smoothed_kl_moderate,
         linewidth=2, color='blue', label='LR = 0.01')

ax2.axhline(y=0.01, color='green', linestyle='--', linewidth=1.5, label=r'$\delta = 0.01$')
ax2.set_xlabel('Episode', fontsize=11)
ax2.set_ylabel('KL Divergence per Update', fontsize=11)
ax2.set_title('KL Divergence per Update', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

print("Left plot: The large LR agent shows high variance and can crash.")
print("Right plot: The large LR agent's KL per update is far above the TRPO threshold.")
print("The moderate LR agent learns more steadily with bounded KL.")

### What the output shows

The large learning rate (0.5) causes erratic performance with high variance. The agent's KL divergence per update is far above the TRPO threshold of 0.01, meaning each step moves the policy so far that the old data becomes unreliable. The moderate learning rate (0.01) learns more steadily with much smaller policy shifts.

**The one sentence to say in an interview:** "Without trust regions, a large gradient step can overshoot, and because the policy generates its own training data, a bad policy collects bad data, creating a vicious cycle that is hard to recover from."

---
## Experiment 3: Clipping Ablation (PPO-Clip vs Vanilla Policy Gradient)

**Claim being tested:** PPO's clipping mechanism limits KL divergence per update without explicitly computing or constraining it. Tighter clipping (smaller epsilon) produces smaller KL divergence.

**Why this matters in an interview:** PPO replaces TRPO's expensive KL constraint with a simple clip. If you can show that clipping achieves bounded KL in practice, you have explained why PPO works and why it replaced TRPO as the industry standard.

**Setup:**
- Implement vanilla policy gradient (no clip), PPO with eps=0.2, and PPO with eps=0.05
- Train each for 20 iterations, collecting 10 trajectories per iteration
- Measure mean KL divergence per update
- Plot KL per iteration for all three methods

In [ ]:
def train_ppo_clip(env, clip_epsilon=None, lr=0.1, n_iterations=20,
                   n_trajectories_per_iter=10, n_gradient_steps=5, seed=42):
    """
    Train an agent using PPO-clip (or vanilla PG if clip_epsilon is None).

    Args:
        clip_epsilon: PPO clipping parameter. None = vanilla PG (no clipping).
        n_gradient_steps: number of gradient steps per batch of data.

    Returns:
        kl_per_iter: KL divergence after each iteration's updates
        reward_per_iter: mean reward per iteration
    """
    np.random.seed(seed)
    torch.manual_seed(seed)

    logits = torch.zeros(env.n_states, env.n_actions, requires_grad=True)
    kl_per_iter = []
    reward_per_iter = []

    for iteration in range(n_iterations):
        old_logits = logits.clone().detach()
        old_probs = torch.softmax(old_logits, dim=-1).detach()

        # Collect batch of trajectories
        all_states, all_actions, all_returns = [], [], []
        total_rewards = []
        for _ in range(n_trajectories_per_iter):
            states, actions, rewards = collect_trajectory(env, logits)
            rets = compute_returns(rewards)
            all_states.extend(states)
            all_actions.extend(actions)
            all_returns.extend(rets)
            total_rewards.append(sum(rewards))

        reward_per_iter.append(np.mean(total_rewards))

        states_t = torch.tensor(all_states, dtype=torch.long)
        actions_t = torch.tensor(all_actions, dtype=torch.long)
        returns_t = torch.tensor(all_returns, dtype=torch.float32)
        returns_t = (returns_t - returns_t.mean()) / (returns_t.std() + 1e-8)

        # Old action probabilities for ratio computation
        old_action_probs = old_probs[states_t, actions_t].detach()

        # Multiple gradient steps on the same batch (PPO style)
        for _ in range(n_gradient_steps):
            probs = torch.softmax(logits, dim=-1)
            new_action_probs = probs[states_t, actions_t]
            ratio = new_action_probs / (old_action_probs + 1e-10)

            if clip_epsilon is not None:
                # PPO-clip objective
                clipped_ratio = torch.clamp(ratio, 1.0 - clip_epsilon, 1.0 + clip_epsilon)
                surrogate = torch.min(ratio * returns_t, clipped_ratio * returns_t)
            else:
                # Vanilla policy gradient (no clipping)
                surrogate = ratio * returns_t

            loss = -surrogate.mean()

            if logits.grad is not None:
                logits.grad.zero_()
            loss.backward()

            with torch.no_grad():
                logits -= lr * logits.grad
            logits.requires_grad_(True)

        # Measure KL after all gradient steps this iteration
        kl = compute_kl_divergence(old_logits, logits.detach())
        kl_per_iter.append(kl)

    return kl_per_iter, reward_per_iter


env = ChainMDP()

print("Training: Vanilla PG (no clip) ...")
kl_vanilla, rew_vanilla = train_ppo_clip(env, clip_epsilon=None, lr=0.1, seed=42)
print(f"  Mean KL per iteration: {np.mean(kl_vanilla):.4f}")
print(f"  Max KL in any iteration: {np.max(kl_vanilla):.4f}")

print("\nTraining: PPO clip eps=0.2 ...")
kl_ppo02, rew_ppo02 = train_ppo_clip(env, clip_epsilon=0.2, lr=0.1, seed=42)
print(f"  Mean KL per iteration: {np.mean(kl_ppo02):.4f}")
print(f"  Max KL in any iteration: {np.max(kl_ppo02):.4f}")

print("\nTraining: PPO clip eps=0.05 ...")
kl_ppo005, rew_ppo005 = train_ppo_clip(env, clip_epsilon=0.05, lr=0.1, seed=42)
print(f"  Mean KL per iteration: {np.mean(kl_ppo005):.4f}")
print(f"  Max KL in any iteration: {np.max(kl_ppo005):.4f}")

print(f"\nVanilla PG max KL is {np.max(kl_vanilla)/np.max(kl_ppo005):.1f}x larger than PPO eps=0.05")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

iterations = range(1, 21)

# Left: KL divergence per iteration
ax1 = axes[0]
ax1.plot(iterations, kl_vanilla, 'o-', linewidth=2, markersize=5,
         color='red', label='Vanilla PG (no clip)')
ax1.plot(iterations, kl_ppo02, 's-', linewidth=2, markersize=5,
         color='orange', label=r'PPO $\epsilon=0.2$')
ax1.plot(iterations, kl_ppo005, '^-', linewidth=2, markersize=5,
         color='blue', label=r'PPO $\epsilon=0.05$')

ax1.axhline(y=0.01, color='green', linestyle='--', linewidth=1.5, label=r'TRPO $\delta = 0.01$')
ax1.set_xlabel('Training Iteration', fontsize=11)
ax1.set_ylabel('KL Divergence (old || new)', fontsize=11)
ax1.set_title('Experiment 3: Clipping Reduces KL Divergence', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Right: Reward per iteration
ax2 = axes[1]
ax2.plot(iterations, rew_vanilla, 'o-', linewidth=2, markersize=5,
         color='red', label='Vanilla PG (no clip)')
ax2.plot(iterations, rew_ppo02, 's-', linewidth=2, markersize=5,
         color='orange', label=r'PPO $\epsilon=0.2$')
ax2.plot(iterations, rew_ppo005, '^-', linewidth=2, markersize=5,
         color='blue', label=r'PPO $\epsilon=0.05$')

ax2.set_xlabel('Training Iteration', fontsize=11)
ax2.set_ylabel('Mean Episode Reward', fontsize=11)
ax2.set_title('Reward per Iteration', fontsize=13, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Left plot: Clipping reduces KL divergence. Tighter clip (eps=0.05) gives smaller KL.")
print("Right plot: All methods improve, but clipped versions are more stable.")
print("\nKey insight: PPO achieves bounded KL without computing KL explicitly.")

In [ ]:
# Summary comparison table
print("="*70)
print("CLIPPING ABLATION SUMMARY")
print("="*70)
print(f"{'Method':<25} {'Mean KL':>10} {'Max KL':>10} {'Final Reward':>14}")
print("-"*70)
print(f"{'Vanilla PG (no clip)':<25} {np.mean(kl_vanilla):>10.4f} {np.max(kl_vanilla):>10.4f} {np.mean(rew_vanilla[-5:]):>14.2f}")
print(f"{'PPO eps=0.2':<25} {np.mean(kl_ppo02):>10.4f} {np.max(kl_ppo02):>10.4f} {np.mean(rew_ppo02[-5:]):>14.2f}")
print(f"{'PPO eps=0.05':<25} {np.mean(kl_ppo005):>10.4f} {np.max(kl_ppo005):>10.4f} {np.mean(rew_ppo005[-5:]):>14.2f}")
print("="*70)
print("\nTighter clipping => smaller KL divergence => more conservative updates.")
print("This is the trade-off: stability vs speed of learning.")

### What the output shows

Clipping the probability ratio effectively bounds the KL divergence per update, even though KL is never explicitly computed or constrained. Tighter clipping (eps=0.05) produces smaller KL divergence than looser clipping (eps=0.2), which in turn produces smaller KL than no clipping at all.

**The one sentence to say in an interview:** "PPO's clip replaces TRPO's explicit KL constraint with a simpler mechanism that achieves the same effect — bounded policy change per update — using only first-order optimization."

---
## Summary: Claims Now Backed by Evidence

| Claim | Experiment | Key Number |
|-------|------------|------------|
| Larger learning rates cause larger KL divergence | Exp 1 | LR 0.5 KL >> LR 0.001 KL |
| Large policy updates cause instability (policy collapse) | Exp 2 | LR 0.5 shows erratic rewards and high KL |
| PPO clipping bounds KL without explicit KL constraint | Exp 3 | Clipped KL < Unclipped KL |
| Tighter clipping gives smaller KL (more conservative) | Exp 3 | eps=0.05 KL < eps=0.2 KL |

**For deeper reading:** see [trust-region-methods-interview.md](./trust-region-methods-interview.md) for the full mathematical treatment, failure modes, and interview Q&A.

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)